# Phase 6 — Serving the model over HTTP

Everything through Phase 5 runs in a notebook on one machine. The model is a
`.pt` file that only a Python process with this repository installed can use.
This phase turns it into a *service*: something with a stable contract that
anything speaking HTTP can call, without knowing it is Python, PyTorch, or a
Coulomb matrix.

**What REST means here, concretely.** Resources get URLs and are acted on with
HTTP verbs. `GET /model` reads a description of what is loaded; `POST /predict`
submits a molecule and gets a number back. `GET` because reading is safe to
repeat; `POST` because a prediction carries a request body and should not be
cached against its URL. Errors travel as status codes, which is what lets a
caller separate *"your request was wrong"* (4xx) from *"the service is broken"*
(5xx) without parsing prose.

**Why FastAPI rather than a bare HTTP handler.** The request shape is declared
once as a typed class, and three things are derived from that one declaration:
the validator that rejects malformed requests, the OpenAPI document behind
`/docs`, and the types the endpoint functions are written against. Hand-rolling
the validation gives only the first, and it drifts out of agreement with the
documentation immediately.

**Why validation matters more for a model than for an ordinary service.** A
malformed request to a database-backed API usually fails loudly. A malformed
request to a *model* does not: hand this network 435 plausible-looking numbers
and it returns a confident value that means nothing. The boundary has to be
strict on purpose, because the failure mode is silence.

## The order of business

1. Open the test split — once, on one model.
2. Write the checkpoint that ships, with that score recorded inside it.
3. Exercise the API.

In [1]:
import json

import numpy as np
import pandas as pd
from fastapi.testclient import TestClient

from molecular_property_predictor.api.main import create_app
from molecular_property_predictor.api.schemas import METHANE_EXAMPLE
from molecular_property_predictor.data import load_qm9
from molecular_property_predictor.model import DEFAULT_MODEL_DIR, load_artifact
from molecular_property_predictor.train import evaluate_on_test, record_test_score

df = load_qm9()

# The winner of the Phase 5 sweep, as confirmed by 05b_convergence.ipynb.
WINNER = DEFAULT_MODEL_DIR / "molecular_net_convergence-le0.001-hi1024x512x256.pt"
SERVED = DEFAULT_MODEL_DIR / "served.pt"

_, _, metadata = load_artifact(WINNER)
print(f"checkpoint      : {WINNER.name}")
print(f"representation  : {metadata.representation}  ({metadata.n_features} features)")
print(f"architecture    : {metadata.hidden_sizes}, dropout {metadata.dropout}")
print(f"split seed      : {metadata.split_seed}")
print(f"validation MAE  : {metadata.validation_mae_ev:.4f} eV  (epoch {metadata.best_epoch})")
print(f"test MAE        : {metadata.test_mae_ev}")

C:\Users\rahul\OneDrive\Desktop\Project\molecular-property-predictor\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


checkpoint      : molecular_net_convergence-le0.001-hi1024x512x256.pt
representation  : sorted_coulomb  (435 features)
architecture    : (1024, 512, 256), dropout 0.1
split seed      : 0
validation MAE  : 0.2476 eV  (epoch 90)
test MAE        : None


## Opening the test split

Every score reported in Phases 3, 4, 5 and 5b is a **validation** score, and each
one informed a choice: which representation, which loss function, which learning
rate, which width, whether the budget was binding. A split used to make choices
can no longer estimate performance on unseen data, because the choices have been
fitted to it. That is why the test rows have not been touched since Phase 1.

The model is now fixed. Nothing below changes it, so this number cannot leak back
into a decision — which is the only condition under which it means what it says.

`evaluate_on_test` deliberately takes no seed and no representation. Both are
read from the artifact's own metadata, so the molecules scored are exactly the
ones this checkpoint never saw. Accepting a seed would make it possible to score
a model against the wrong split, which would not raise and would simply report a
number better than the truth.

In [2]:
test_scores = evaluate_on_test(df, WINNER)

comparison = pd.DataFrame([
    {"split": "validation", "mae_ev": metadata.validation_mae_ev, "note": "used for every choice"},
    {"split": "test", "mae_ev": test_scores["mae_ev"], "note": "opened once, here"},
])

display(comparison.style.format({"mae_ev": "{:.4f}"}).hide(axis="index"))
print(f"test MAE   {test_scores['mae_ev']:.4f} eV")
print(f"test RMSE  {test_scores['rmse_ev']:.4f} eV")
print(f"test R2    {test_scores['r2']:.4f}")
print(f"\ngap vs validation: {test_scores['mae_ev'] - metadata.validation_mae_ev:+.4f} eV")

split,mae_ev,note
validation,0.2476,used for every choice
test,0.2439,"opened once, here"


test MAE   0.2439 eV
test RMSE  0.3571 eV
test R2    0.9222

gap vs validation: -0.0037 eV


### Reading that gap

A test score close to the validation score is the expected outcome here, and it
is worth being precise about *why* rather than treating it as a pass mark.

The two splits are random draws from the same 130,831 molecules, so they are
identically distributed by construction. What the gap measures is how much the
choices made on validation were fitted to that particular sample of 13,083
molecules. Nine sweep configurations plus a handful of earlier decisions is very
little selection pressure — so a small gap is what the design predicts, not a
lucky result.

**What this number does not establish.** The split is random, which means a
molecule in the test set usually has close structural relatives in the training
set. That flatters any model here relative to how it would perform on a genuinely
novel scaffold, and it has been the honest caveat since Phase 1. A scaffold-based
or size-extrapolation split would be the harder and more informative test, and
this project does not do one.

## The checkpoint that ships

The score is written *inside* the artifact rather than into a separate file, for
the same reason the scaler is: a number that has to travel alongside the weights
will eventually be separated from them. `GET /model` reports it, so anyone
calling the API can see the model's measured error without reading this
repository.

The served checkpoint gets a fixed name. Which model is served is a decision,
made once and recorded — not something rediscovered at start-up from whichever
file in `models/` happens to sort first.

In [3]:
served_path = record_test_score(WINNER, test_scores["mae_ev"], destination=SERVED)

_, _, served_metadata = load_artifact(served_path)
print(f"wrote {served_path.name} ({served_path.stat().st_size / 1e6:.1f} MB)")
print(f"  validation MAE : {served_metadata.validation_mae_ev:.4f} eV")
print(f"  test MAE       : {served_metadata.test_mae_ev:.4f} eV")

wrote served.pt (4.4 MB)
  validation MAE : 0.2476 eV
  test MAE       : 0.2439 eV


## Exercising the API

`TestClient` drives the application in-process — no port is opened and no server
runs. The routing, validation and serialisation are the real ones, so this
behaves identically to a deployed instance.

To run it as an actual server instead:

```bash
uvicorn molecular_property_predictor.api.main:app --reload
```

FastAPI is a framework and does not own a socket; uvicorn is the process that
accepts connections and speaks HTTP. Keeping them separate is what lets the app
be tested without a network, as it is here.

In [4]:
client = TestClient(create_app(SERVED))
client.__enter__()  # triggers the lifespan handler, which loads the model

print("GET /health")
print(json.dumps(client.get("/health").json(), indent=2))

print("\nGET /model")
print(json.dumps(client.get("/model").json(), indent=2))

GET /health
{
  "status": "ok",
  "model_loaded": true
}

GET /model
{
  "target": "lumo",
  "units": "eV",
  "representation": "sorted_coulomb",
  "n_features": 435,
  "n_parameters": 1102849,
  "hidden_sizes": [
    1024,
    512,
    256
  ],
  "dropout": 0.1,
  "loss_name": "mse",
  "split_seed": 0,
  "torch_seed": 0,
  "epochs_trained": 100,
  "best_epoch": 90,
  "validation_mae_ev": 0.2476282933006721,
  "test_mae_ev": 0.24390529686433385,
  "max_atoms": 29,
  "supported_elements": [
    1,
    6,
    7,
    8,
    9
  ]
}


### One prediction

Methane, taken verbatim from `dsgdb9nsd_000001.xyz`. Its reference LUMO is in the
dataset, so the prediction can be checked against the value the quantum chemistry
calculation produced — noting that methane is in *some* split, and which one
depends on the seed.

In [5]:
response = client.post("/predict", json=METHANE_EXAMPLE)
print("POST /predict")
print(json.dumps(response.json(), indent=2))

reference = df.loc[df["index"] == 1, "lumo"].iloc[0]
predicted = response.json()["lumo_ev"]
print(f"\nreference (QM9, B3LYP) : {reference:+.4f} eV")
print(f"predicted              : {predicted:+.4f} eV")
print(f"error                  : {predicted - reference:+.4f} eV")

POST /predict
{
  "lumo_ev": 2.0034375190734863,
  "units": "eV",
  "n_atoms": 5
}

reference (QM9, B3LYP) : +3.1865 eV
predicted              : +2.0034 eV
error                  : -1.1830 eV


### That error is five times the model's average. Why it is kept here

1.18 eV against a test MAE of 0.244 eV looks like a broken demo. It is not, and
swapping methane for a molecule that scores well would hide the single most
useful thing this notebook can show.

Methane is the most extreme molecule in QM9. Its LUMO of +3.19 eV sits at the
**100th percentile** — only two molecules out of 130,831 exceed 3.0 eV — and it
is one of just five molecules in the entire dataset with five atoms. It is about
as far from the training distribution's centre as a QM9 molecule can be while
still being in QM9.

**And it is in the training split.** The model saw this molecule and still misses
it by 1.18 eV, pulled toward the mean of +0.32 eV. That is worth sitting with,
because the intuitive reading — "it should have memorised it" — is backwards. A
network minimising mean squared error across 104,664 molecules will not distort
itself to fit two outliers; the loss it would pay everywhere else exceeds what it
saves on them. Failing to memorise a training point is here a sign of a model
that generalises rather than one that is broken.

This is the shrinkage measured in Phase 3, where the predicted-versus-true slope
came out at 0.811 instead of 1.0, and which Phase 4's network improved to roughly
0.91 without removing. The batch below, eight molecules drawn at random with 12
to 17 atoms, averages 0.19 eV of error — that is the behaviour to expect in the
bulk of the distribution.

**The practical consequence, which the API does not announce:** predictions near
the edges of the LUMO range are systematically pulled inward, so the molecules
most likely to be interesting in a screening application are exactly the ones
this model is least reliable about. That belongs in the README, not only here.

### A batch

One forward pass over many molecules rather than one request each. For a network
this small the per-request overhead — HTTP, validation, moving a tensor to the
device — is a large fraction of the cost, so screening a candidate library one
request at a time would spend most of its time on everything except the model.

In [6]:
sample = df.sample(8, random_state=0)
molecules = [
    {
        # int(), not list(): the frame holds numpy int64, which json cannot
        # serialise. A real client sending JSON never hits this -- it is an
        # artefact of building a request from a DataFrame.
        "atomic_numbers": [int(z) for z in row["atomic_numbers"]],
        "coordinates": np.asarray(row["coordinates"]).reshape(-1, 3).tolist(),
    }
    for _, row in sample.iterrows()
]

batch = client.post("/predict/batch", json={"molecules": molecules}).json()

display(
    pd.DataFrame({
        "n_atoms": [p["n_atoms"] for p in batch["predictions"]],
        "predicted_ev": [p["lumo_ev"] for p in batch["predictions"]],
        "reference_ev": sample["lumo"].to_numpy(),
    }).assign(error_ev=lambda d: d["predicted_ev"] - d["reference_ev"])
    .style.format({"predicted_ev": "{:+.4f}", "reference_ev": "{:+.4f}", "error_ev": "{:+.4f}"})
    .set_caption("Eight molecules in one request")
)

,n_atoms,predicted_ev,reference_ev,error_ev
0,13,-2.4677,-2.7320,+0.2644
1,17,+1.3327,+1.3905,-0.0578
2,17,+0.7781,+0.9252,-0.1471
3,12,-1.6531,-1.4749,-0.1782
4,14,-0.4769,+0.0871,-0.5640
5,15,+0.5438,+0.4354,+0.1084
6,16,-2.2116,-2.0245,-0.1871
7,17,+0.4855,+0.4898,-0.0043


### Refusing what it cannot answer

The interesting half of the contract. Each of these would otherwise reach the
featuriser, and the last two would return a confident number rather than an
error — the coincident-atom case divides by zero and propagates `inf` straight
through the network.

In [7]:
bad_requests = {
    "mismatched lengths": {"atomic_numbers": [6, 1], "coordinates": [[0.0, 0.0, 0.0]]},
    "element outside QM9": {
        "atomic_numbers": [6, 15],
        "coordinates": [[0.0, 0.0, 0.0], [1.5, 0.0, 0.0]],
    },
    "30 atoms (max is 29)": {
        "atomic_numbers": [1] * 30,
        "coordinates": [[i * 1.5, 0.0, 0.0] for i in range(30)],
    },
    "two atoms in one place": {
        "atomic_numbers": [6, 1],
        "coordinates": [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]],
    },
}

for label, payload in bad_requests.items():
    reply = client.post("/predict", json=payload)
    message = reply.json()["detail"][0]["msg"].replace("Value error, ", "")
    print(f"{label:24} -> {reply.status_code}  {message[:88]}")

client.__exit__(None, None, None)

mismatched lengths       -> 422  2 atomic numbers but 1 coordinates: one entry per atom is required
element outside QM9      -> 422  unsupported atomic numbers [15]; this model was trained on QM9, which contains only [1, 
30 atoms (max is 29)     -> 422  List should have at most 29 items after validation, not 30
two atoms in one place   -> 422  atoms 0 and 1 are 0 A apart, below the 0.1 A floor; the Coulomb matrix divides by this d


## Where Phase 6 leaves the project

The model is reachable over HTTP, the contract is typed and enforced, and the
served checkpoint carries its own provenance and its measured error.

**The honest scope statement**, which belongs next to any number this service
returns: QM9's reference values were computed at B3LYP/6-31G(2df,p) *relaxed*
geometries, and this model was trained on those. It therefore replaces the
quantum-chemistry property calculation, not the geometry optimisation that comes
before it. Submitting a rough or force-field geometry will degrade accuracy
silently — nothing in the response will indicate it. Published QM9 LUMO models
reach roughly 0.02–0.04 eV using message-passing graph networks that learn the
representation and the model together; this is about an order of magnitude
behind, which is the cost of a fixed 2012 descriptor and is not something a
wider network would close.

**What is deliberately absent:** no authentication, no rate limiting, no
per-prediction uncertainty, no request queueing. On uncertainty specifically —
what exists is a population-level MAE, not a per-prediction error bar. Returning
it on every response would read as one, so it appears in `GET /model` instead,
where it is plainly a property of the model rather than of the answer.

**Next:** Phase 7 puts this in a container. One decision it inherits — `models/`
is gitignored, so a build from a clean clone has no checkpoint to serve. The
options are to commit one blessed artifact or fetch it at build time, and that
gets decided there rather than assumed here.